# Address and read

**Rung 1 of 5 (next: rung 2, [04, Keep what you need](04_keep_what_you_need.ipynb);
[03, See it](03_see_it.ipynb) is the picture at this rung). You are here: the trace is a dict of
records, and you want a number out of it.**
Trigger: you want a number out.

Every operation the forward ran has a label, and `trace[label]` is its record: the activation,
its shape and dtype, the function that made it, the module it ran in, its neighbours in the
dataflow and the source line that wrote it. This notebook is the label grammar, the keys that
reach a record, what a record carries, and what happens when a model returns a dict instead of a
tensor.

> **You'll want to know:** [`tl.trace` and the summary](01_capture_and_look.ipynb) (one call
> records a forward pass), [the Trace is read like a dict](00_quickstart.ipynb) (a label in
> square brackets), and that a PyTorch model may return a dict or a tuple, not only a tensor.

In [ ]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [ ]:
from learn_models import tiny_dict_model, tiny_mlp, tiny_rnn  # tiny seeded CPU models

model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

The ReLU's record, and the activation itself.

In [ ]:
trace = tl.trace(model, x)
relu = trace["relu_1_2"]
print(relu.layer_label, tuple(relu.shape), relu.func_name)
print("parents", relu.parents, "| children", relu.children)
relu.out[0, :6].detach()  # the activation: first batch item, first six units

## 1. The label grammar

A layer label has three parts. `relu_1_2` is a `relu`, the first relu in the model, the second
operation overall. Op labels add a pass suffix, `relu_1_2:1`, which matters for layers that run
more than once (section 4). The trace keeps both lists, and three accessors group the records:
`trace.layers` by place in the code, `trace.ops` by execution, `trace.modules` by module.
`count_by` tallies a collection by a field; the input and output records have kind `none`, they
are bookkeeping, not computation.

In [ ]:
print("layers:", trace.layer_labels)
print("ops   :", trace.op_labels)
trace.layers.count_by("kind")

## 2. Four keys that reach the same record

Square brackets take a full label, a position, any unique substring, or a pass-qualified op
label. A module name is a key too: it gives that module's output operation, because what you
usually want from a module is what came out of it. The module itself is a different record, in
`trace.modules`: its class, how often it was called, and the layers that ran inside it.

In [ ]:
keys = [trace["relu_1_2"], trace[2], trace["relu"], trace.ops["relu_1_2:1"]]
print("one record, four keys:", [op.layer_label for op in keys])
first = trace["in_proj"]  # a module name gives that module's output op
print("in_proj's output op:", first.layer_label, "| ran in module", first.module)
module = trace.modules["in_proj"]  # the Module record itself
print(
    "the module:", module.class_name, "| calls:", module.num_calls, "| layers:", module.layer_labels
)

## 3. A teaching error

A key that matches two records refuses rather than guess, and names both. A key that matches
nothing says so and suggests the closest labels. `find_layers` is the search you reach for when
you half-remember a name.

In [ ]:
try:
    trace["linear"]
except tl.errors.AmbiguousOpLookupError as error:
    print(error)
try:
    trace["relu_9"]
except tl.errors.InvalidArgumentError as error:
    print(error)
trace.find_layers("lin")

## 4. Passes

A layer is one place in the code. When that place runs again, in a loop, the layer has several
passes and each pass is its own op: `linear_1_1` is the layer with all three, `linear_1_1:2` the
second pass alone. Each layer's `num_passes` says how often it ran.

In [ ]:
rnn, seq = tiny_rnn()  # one linear cell and a tanh, applied three times in a Python loop
loop = tl.trace(rnn, seq)
cell = loop["linear_1_1"]
print(
    type(cell).__name__,
    "with",
    cell.num_passes,
    "passes |",
    {layer.layer_label: layer.num_passes for layer in loop.layers},
)
print("pass 2:", loop["linear_1_1:2"].out[0, :4].detach())
print("pass 3:", loop["linear_1_1:3"].out[0, :4].detach())

## 5. What a record carries

Beyond the activation, a record carries its dtype, its cost (forward FLOPs and the bytes the
activation takes), and its neighbours: `parents` and `children` are labels, `get_parents()` gives
the records. Captured with `CaptureOptions(save_source_context=True)`, the trace also knows where the
model's code lives, and each record knows the call site that produced it.

In [ ]:
print(
    "dtype",
    relu.dtype,
    "| flops",
    relu.cost.flops_forward,
    "| activation bytes",
    relu.cost.activation_memory,
)
print("first parent record:", relu.get_parents()[0].layer_label)
sourced = tl.trace(model, x, capture=tl.options.CaptureOptions(save_source_context=True))
print(
    "model source:",
    Path(sourced.root_module.class_source_file).name,
    "| forward at line",
    sourced.root_module.forward_source_line,
)
sourced["relu_1_2"].code_context[0]  # the call site that produced this op

To keep the source text itself with the trace, ask for it with one more keyword on the same
call. `capture=` takes a `CaptureOptions`; this is the first of several options you will meet
there.

In [ ]:
with_source = tl.trace(model, x, capture=tl.options.CaptureOptions(save_source_context=True))
with_source["relu_1_2"].code_context[0]

## 6. Containers: a model that returns a dict

Many models return a dict, a tuple or a Hugging Face `ModelOutput`. The trace keeps the
structure. `trace.reconstruct_output()` rebuilds the very object the model returned, with the saved
activations as its leaves, and `trace.output_layers` names the operations that fed it. The
output record's `container` is a view of that structure: index it by key to reach the leaf
record and its activation, or `reconstruct()` the whole object from the saved leaves.

In [ ]:
dict_model, dx = tiny_dict_model()  # returns {"logits": ..., "hidden": ...}
dt = tl.trace(dict_model, dx)
print(
    "trace.reconstruct_output() rebuilds the dict:",
    {k: tuple(v.shape) for k, v in dt.reconstruct_output().items()},
)
print("output layers:", dt.output_layers)
view = dt[dt.output_layers[0]].container
print(
    "container:", view.kind, "with", len(view), "entries | 'logits' is", view["logits"].layer_label
)
print("reconstructed:", {k: tuple(v.shape) for k, v in view.reconstruct().items()})
view["logits"].out[0].detach()  # the leaf record's activation: first item's three logits

## 7. When to switch layers

TorchLens has two layers. The one-call functions, `tl.summary`, `tl.pluck` and `tl.extract`,, each run a forward and hand back one thing. The `Trace` is the object layer: one
forward, read as often as you like. Switch to the trace the moment you want two things from the
same forward pass. Matplotlib users know this move as going from `plt.plot` to `ax.plot`.

## 8. Where this leaves you

Every operation has a label you can read, four ways to type it, and a record that carries the
activation, its cost, its neighbours and its source line; a model that returns a dict keeps the
dict. The next rungs add keywords to `tl.trace` that choose what to keep and what to change.

**One thing about networks:** the ReLU's `parents` and `children` are the dataflow, not the
module tree. The model's code is a stack of modules, but what it computes is a graph of tensors;
TorchLens addresses the graph first and the stack second, which is why a module name gives you an
output tensor.

**Go deeper:** next rung, [04, Keep what you need](04_keep_what_you_need.ipynb), or
[03, See it](03_see_it.ipynb) to draw the graph first. Recipe cards:
[`pluck.ipynb`](../../examples/5min/pluck.ipynb),
[`extract_activations.ipynb`](../../examples/5min/extract_activations.ipynb). Reference: the
glossary entries [Op, Layer and Module](../../docs/reference/glossary.md#graph-records) and
[the grouping guide](../../docs/guides/grouping.md) for the exact pass rule. A return type
TorchLens does not know yet is taught with `tl.types.register_container(MyType, flatten,
unflatten)`; [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) shows one.